# Notebook 04: Segmentation and Heterogeneous Treatment Effects

## Research question

Does the effect of the Men's email differ systematically across pre-specified customer segments?

## Analytical objective

Estimate treatment effects within segments defined by customer history, channel, recency, geography, product affinity, and tenure; then distinguish genuine effect modification from sampling variation.

## Statistical framework

The primary comparison is `Mens E-Mail` versus `No E-Mail`. Subgroup estimates are summarized with 95% confidence intervals. Formal evidence of heterogeneity comes from treatment-by-segment interaction terms estimated with HC3 robust standard errors and Benjamini-Hochberg adjustment.

## Required output

Complete the TODO cells, produce the subgroup effect tables and forest plots, identify any adjusted-significant interactions, and state whether segment-specific targeting is justified.

In [1]:
import pandas as pd
import numpy as np

import statsmodels.formula.api as smf
from statsmodels.stats import multitest
import matplotlib.pyplot as plt

In [2]:
hillstrom_df = pd.read_csv("../data/raw/hillstrom.csv")

hillstrom_df

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0
63998,1,5) $500 - $750,552.94,1,0,Surburban,1,Multichannel,Womens E-Mail,0,0,0.0


In [3]:
mens_control_df = hillstrom_df[hillstrom_df["segment"].isin(["No E-Mail", "Mens E-Mail"])].copy()

mens_control_df["recency_group"] = np.where(mens_control_df["recency"] < mens_control_df["recency"].median(), "recent", "non-recent")

mens_control_df["history_group"] = mens_control_df["history_segment"].map({
    '1) $0 - $100': 'low',
    '2) $100 - $200': 'mid',
    '3) $200 - $350': 'mid',
    '4) $350 - $500': 'high',
    '5) $500 - $750': 'high',
    '6) $750 - $1,000': 'high',
    '7) $1,000 +': 'high'
})

mens_control_df

,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend,recency_group,history_group
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0,non-recent,mid
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0,non-recent,high
8,9,5) $500 - $750,675.07,1,1,Rural,1,Phone,Mens E-Mail,0,0,0.0,non-recent,high
13,2,2) $100 - $200,101.64,0,1,Urban,0,Web,Mens E-Mail,1,0,0.0,recent,mid
14,4,3) $200 - $350,241.42,0,1,Rural,1,Multichannel,No E-Mail,0,0,0.0,recent,mid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63994,7,1) $0 - $100,86.46,0,1,Urban,0,Web,Mens E-Mail,0,0,0.0,non-recent,low
63995,10,2) $100 - $200,105.54,1,0,Urban,0,Web,Mens E-Mail,0,0,0.0,non-recent,mid
63996,5,1) $0 - $100,38.91,0,1,Urban,1,Phone,Mens E-Mail,0,0,0.0,recent,low
63997,6,1) $0 - $100,29.99,1,0,Urban,1,Phone,Mens E-Mail,0,0,0.0,non-recent,low


In [4]:
# TODO: Implement `compute_effects`.
# For every covariate and level, calculate the Mens-vs-Control mean difference,
# its standard error, and a 95% confidence interval. Return one dictionary per
# subgroup with cov, level, sample sizes, effect, ci_low, and ci_high.
def compute_effects(outcome, cov_labels, df):
    results = []

    for cov in cov_labels:
        # Compute subgroup means, standard errors, and counts once per covariate.
        grouped = df.groupby([cov, "segment"])[outcome].agg(
            mean="mean",
            sem="sem",
            count="count"
        )

        for level in sorted(df[cov].dropna().unique()):
            treated = grouped.loc[(level, "Mens E-Mail")]
            control = grouped.loc[(level, "No E-Mail")]

            effect = treated["mean"] - control["mean"]
            se = np.sqrt(treated["sem"] ** 2 + control["sem"] ** 2)

            results.append({
                "cov": cov,
                "level": level,
                "n_treated": int(treated["count"]),
                "n_control": int(control["count"]),
                "effect": effect,
                "ci_low": effect - 1.96 * se,
                "ci_high": effect + 1.96 * se,
            })

    return results

# TODO: Implement `forest_plot` or use the provided plotting logic to show
# point estimates, 95% CIs, and a vertical zero-effect line.
def forest_plot(results, cov_labels, outcome_label, ax):
    # Keep covariate headers and subgroup rows separate on the y-axis.
    y_positions = []
    y_labels = []
    header_ys = set()
    plot_items = []
    current_y = 0
    previous_cov = None

    for result in results:
        cov = result["cov"]

        if cov != previous_cov:
            if previous_cov is not None:
                current_y -= 0.4

            y_positions.append(current_y)
            y_labels.append(cov_labels[cov])
            header_ys.add(current_y)
            current_y -= 1
            previous_cov = cov

        y_positions.append(current_y)
        y_labels.append(f"  {result['level']}")
        plot_items.append((current_y, result))
        current_y -= 1

    for y, result in plot_items:
        ax.errorbar(
            result["effect"],
            y,
            xerr=[[result["effect"] - result["ci_low"]],
                  [result["ci_high"] - result["effect"]]],
            fmt="o",
            color="black",
            capsize=4,
            markersize=5,
        )

    ax.axvline(0, color="red", linestyle="--", alpha=0.7)
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("Treatment Effect (Mens vs Control)\n"
                  "Point = mean difference, bars = 95% CI")
    ax.set_title(outcome_label)

    for tick, y in zip(ax.get_yticklabels(), y_positions):
        if y in header_ys:
            tick.set_fontweight("bold")

# Plot every pre-specified covariate for all three outcomes.
cov_labels = {
    "channel": "Channel",
    "recency_group": "Recency",
    "history_group": "Prior Spend",
    "mens": "Mens",
    "womens": "Womens",
    "zip_code": "Zip Code",
    "newbie": "New Customer",
}

missing_covariates = [
    cov for cov in cov_labels if cov not in mens_control_df.columns
]
if missing_covariates:
    raise ValueError(f"Missing covariates: {missing_covariates}")

outcome_labels = {
    "visit": "Visit Rate",
    "conversion": "Conversion Rate",
    "spend": "Spend ($)",
}

for outcome, outcome_label in outcome_labels.items():
    results = compute_effects(
        outcome=outcome,
        cov_labels=cov_labels,
        df=mens_control_df,
    )

    results_df = pd.DataFrame(results)
    results_df["n_total"] = (
        results_df["n_treated"] + results_df["n_control"]
    )
    display(results_df[[
        "cov", "level", "n_treated", "n_control", "n_total",
        "effect", "ci_low", "ci_high",
    ]].round(4))

    # One row per subgroup plus one header row per covariate.
    n_plot_rows = len(results) + len(cov_labels)
    fig_height = max(7, 0.42 * n_plot_rows)
    fig, ax = plt.subplots(figsize=(9, fig_height))
    forest_plot(results, cov_labels, outcome_label, ax)
    fig.tight_layout()
    fig.savefig(
        f"../reports/figures/forest_plot_{outcome}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()
    plt.close(fig)


### Test whether subgroup effects truly differ

The model `outcome ~ treatment * C(segment)` contains an interaction term. The treatment main effect is the effect in the reference group; the interaction term asks how another segment differs from that reference effect. We use HC3 robust standard errors and adjust all interaction p-values together.

In [ ]:
# TODO: Fit `outcome ~ treatment * C(covariate)` for every outcome and
# covariate. Collect only interaction-term p-values, use HC3 robust standard
# errors, then apply Benjamini-Hochberg across all interaction tests.
analysis_df = mens_control_df.copy()
analysis_df["treatment"] = (
    analysis_df["segment"] == "Mens E-Mail"
).astype(int)

interaction_results = []

for outcome in outcome_labels:
    for covariate in cov_labels:
        model = smf.ols(
            formula=f"{outcome} ~ treatment * C({covariate})",
            data=analysis_df,
        ).fit(cov_type="HC3")

        for term in model.pvalues.index:
            is_interaction = (
                "treatment" in term
                and f"C({covariate})" in term
                and ":" in term
            )
            if is_interaction:
                interaction_results.append({
                    "outcome": outcome,
                    "covariate": covariate,
                    "term": term,
                    "estimate": model.params[term],
                    "std_error_hc3": model.bse[term],
                    "p_value": model.pvalues[term],
                })

heterogeneity_df = pd.DataFrame(interaction_results)

reject, adjusted_p_values, _, _ = multitest.multipletests(
    heterogeneity_df["p_value"],
    alpha=0.05,
    method="fdr_bh",
)
heterogeneity_df["p_value_bh"] = adjusted_p_values
heterogeneity_df["significant_bh"] = reject
heterogeneity_df = heterogeneity_df.sort_values(
    ["p_value_bh", "p_value"],
).reset_index(drop=True)

display(heterogeneity_df.round(4))

significant_interactions = heterogeneity_df[
    heterogeneity_df["significant_bh"]
]
if significant_interactions.empty:
    print("No interaction remains significant after BH adjustment.")
else:
    display(significant_interactions.round(4))

## Targeting recommendation

The Men's Email produced positive estimated lifts in visit rate, conversion rate, and spend across the examined customer subgroups. Some point estimates appear larger for particular groups—for example, customers with high prior spend—but these are descriptive differences and should not be interpreted as evidence that those groups respond differently.

None of the 30 treatment-by-covariate interaction terms remained significant after Benjamini–Hochberg adjustment. The smallest adjusted p-value was 0.406, so the analysis does not provide statistically reliable evidence of heterogeneous treatment effects by channel, recency, prior spend, product affinity, geography, or customer tenure.

**Recommendation:** use a broad rollout rather than segment-specific targeting. Restricting the campaign to the subgroups with the largest observed point estimates would rely on sampling noise and could exclude customers who also benefit. If operational constraints require targeting, treat the apparent subgroup differences as hypotheses and validate them in a new experiment designed and powered to test those interactions directly. Continue monitoring overall incremental lift, campaign cost, and downstream value during rollout.
